# ZestXML vs SASRec — Amazon Reviews 2023

One file, no library. The cell below downloads it; open `zestxml_vs_sasrec_standalone.py`
in the Colab file browser to edit anything — the pattern miner, the bilinear scorer, the
SASRec block, the split, the metric.

**Task is extreme multi-label, not next-item.** A point is a user; its labels are the set of
items touched in the next `HORIZON` steps. Metrics are P@k / nDCG@k / propensity-scored
PSP@k, same definitions as `zestxml/eval.py`.

**Why Amazon and not ml-1m.** ml-1m was close to a worst case for a text-scored model, and
the `profile()` line printed at the start of every run says so in numbers: median 112
interactions per item, only 13% of items under 10, and the average item's title tokens
shared by **449 other items** (`Toy Story (1995) Animation Children's Comedy` is three genre
words). A label feature bag cannot identify anything under those conditions. Product titles
are genuinely discriminative and the catalogue has a real tail.

**Read the profile line before the table.** If "mean over items" comes back in the tens
rather than the hundreds, the text carries identity and the comparison is fair.

In [ ]:
!wget -q -O zestxml_vs_sasrec_standalone.py https://raw.githubusercontent.com/hanialshater/zestxml/claude/pytorch-rewrite-fhuwl4/benchmarks/colab/zestxml_vs_sasrec_standalone.py
!pip -q install scikit-learn > /dev/null
import torch, os
print('file:', os.path.getsize('zestxml_vs_sasrec_standalone.py'), 'bytes')
print(torch.__version__, 'cuda' if torch.cuda.is_available() else 'CPU only')

In [ ]:
src = open('zestxml_vs_sasrec_standalone.py').read()
exec(compile(src.replace('if __name__ == "__main__":\n    main()', ''), 'standalone', 'exec'))
print('loaded:', sorted(LOADERS))

## Configuration

`MAX_USERS` matters. Both models produce a dense `(n_users, n_items)` score matrix, so
users x items is the memory wall — 5000 x 30000 is about 600 MB per arm. Start small,
confirm the profile looks right, then raise it.

Categories worth trying, smallest first: `Musical_Instruments`, `Video_Games`,
`Office_Products`, `Toys_and_Games`.

In [ ]:
DATASET       = 'amazon'          # 'amazon' or 'ml-1m'
CATEGORY      = 'Video_Games'
N_CATS        = 0        # category tokens appended to a title. Amazon's list runs generic
                         # -> specific, so anything from the front puts "Video Games" on
                         # every item and buries the title. 0 = title only, 1 = most specific.

MAX_USERS     = 200000   # users READ from the file
MIN_USER      = 8        # k-core: minimum interactions per user ...
MIN_ITEM      = 20       # ... and per item. This is what stops the catalogue being mostly
                         # items nobody touched. Watch the profile line move as you raise it.
MAX_TEST_USERS = 4000    # users SCORED. Decoupled from the above: the dense score matrix is
                         # the memory wall, and shrinking the dataset to fit it is what
                         # broke the first Amazon run.

COLD_FRAC     = 0.05     # on a sparse catalogue 0.1 takes most of the test set with it --
                         # the run warns if over 25% of test positives land on cold items
HORIZON       = 3        # future window forming a point's labels; needs 2*H+2 per user
SHORTY_K      = 500
SASREC_EPOCHS = 200
ZEST_EPOCHS   = 20
CTX           = 20
WINDOWS       = 8
KEEP_FRAC     = 1.0
SEED          = 0

## Run

`random` and `popularity` rows come first and are not decoration: the group rows mask
labels, so the cold row asks "rank the N cold items", which has a floor well above zero. On
ml-1m that floor was P@1 0.31 and both ZestXML (1.07) and plain SASRec (0.97) sat around 3x
it — a much weaker result than either number looks. **Any cold number that does not clearly
beat the `popularity` row is measuring popularity, not transfer.**

In [ ]:
rows = main(
    dataset=DATASET, category=CATEGORY, max_users=MAX_USERS, n_cats=N_CATS,
    min_user=MIN_USER, min_item=MIN_ITEM, max_test_users=MAX_TEST_USERS,
    cold_frac=COLD_FRAC, horizon=HORIZON, shorty_k=SHORTY_K,
    sasrec_epochs=SASREC_EPOCHS, zest_epochs=ZEST_EPOCHS,
    ctx=CTX, windows=WINDOWS, keep_frac=KEEP_FRAC, seed=SEED,
    arms=('random', 'popularity', 'zestxml', 'sasrec', 'sasrec+content'),
)

## If the download fails

The Amazon host was never reachable from the machine this code was written on, so the URLs
are untested while the parsing is tested. If `main` raises, fetch the two files by hand from
https://amazon-reviews-2023.github.io/ into `raw/amazon-<CATEGORY>/`:

* `<CATEGORY>.csv.gz` — the 5-core **rating only** CSV
* `meta_<CATEGORY>.jsonl.gz` — the raw category metadata, which is where titles live

Then re-run the cell above. `DATASET = 'ml-1m'` is verified end to end and needs no manual
step, if you want a working baseline first.

## What to check before believing the table

1. **The profile line.** Interactions per item, % under 10, and how many items share a
   token. This decides whether the comparison is fair at all.
2. **ZestXML's test shortlist recall.** It is a hard ceiling on every ZestXML metric — on
   ml-1m it was 48.88%, meaning it could not retrieve more than half the positives however
   well it ranked. Raise `SHORTY_K` until it plateaus before blaming the model.
3. **The `popularity` row**, per the note above.
4. **SASRec's loss curve.** Still falling at epoch 200 means the head column is understated.

In [ ]:
# is ZestXML retrieval-bound? sweep the candidate budget, ZestXML only.
for k in (500, 1500, 4000):
    print(f'--- SHORTY_K={k}')
    main(dataset=DATASET, category=CATEGORY, max_users=MAX_USERS, n_cats=N_CATS,
         min_user=MIN_USER, min_item=MIN_ITEM, max_test_users=MAX_TEST_USERS,
         cold_frac=COLD_FRAC, horizon=HORIZON, shorty_k=k,
         zest_epochs=ZEST_EPOCHS, ctx=CTX, windows=WINDOWS, seed=SEED, arms=('zestxml',))

In [ ]:
# does the ordering flip as items get sparser? holds catalogue and text fixed.
for kf in (1.0, 0.5, 0.2):
    print(f'=== KEEP_FRAC={kf}')
    main(dataset=DATASET, category=CATEGORY, max_users=MAX_USERS, n_cats=N_CATS,
         min_user=MIN_USER, min_item=MIN_ITEM, max_test_users=MAX_TEST_USERS,
         cold_frac=COLD_FRAC, horizon=HORIZON, shorty_k=SHORTY_K, keep_frac=kf,
         sasrec_epochs=SASREC_EPOCHS, zest_epochs=ZEST_EPOCHS,
         ctx=CTX, windows=WINDOWS, seed=SEED,
         arms=('popularity', 'zestxml', 'sasrec+content'))